# ImageVideoToVideo_WanVACE_Colab — ảnh người + video mẫu → video, Wan2.1 VACE trên T4

Port từ `Wan2_1_VACE_&_CausVid_LoRA_4_Image+ControlVideo_to_Video` của [Isi-dev](https://github.com/Isi-dev/Google-Colab_Notebooks).

**Khác hai notebook kia**: model không bịa chuyển động. Nó trích khung xương (dwpose) từ từng khung của **video mẫu** rồi vẽ người trong **ảnh** theo đúng tư thế đó — nên chuyển động là của người thật. Đổi lại app phải có sẵn bộ video mẫu.

## Đã đo (v1, v2)

| | |
|---|---|
| v1 / v2 trên T4 | 339 s và 287 s cho 41 khung 432×768, Q4_K_M |
| dwpose | v1 tốn 86 s trên CPU → v2 chạy torchscript trên GPU |
| nội suy | v1 sinh 4 khung/giây RIFE ×6 làm tay vẫy bị "ma" → **v2 mặc định 8 khung/giây, RIFE ×3** (đổi lại bằng `khung_moi_giay=4`) |

Giữ đúng thời gian thật (bản gốc tua nhanh 15 → 30 fps).

## Lỗi v3 đang tìm cách chữa: **TAY TRONG SUỐT**

Ở `vace-v2-j1`, chữ trên bảng quảng cáo (*WELCOME TO / COSMIC GULCH*) **hiện xuyên qua lòng bàn tay**; cẳng tay gần như mất.

**Đã loại trừ RIFE**: ba bản (v1 4fps RIFE ×6, v1 8fps RIFE ×3, v2 8fps) trong suốt như nhau → lỗi nằm trong latent model sinh ra, không phải khâu nội suy.

**Cơ chế** (đọc từ `comfy_extras/nodes_wan.py`): ta gọi `WanVaceToVideo` với `control_masks=None`, nên `mask` toàn 1 → `inactive` = xám trơn, `reactive` = đúng bộ xương dwpose. Tức **toàn khung hình là vùng phải sinh mới**, và tín hiệu duy nhất nói *"chỗ này có cánh tay"* là một bộ xương que trên nền đen. Không silhouette, không depth, không alpha. Ở 4 bước + cfg 1 + Q4, model không giải được mâu thuẫn *"tay ở đây"* vs *"bảng quảng cáo ở đây"* nên lấy trung bình — và trung bình hai lớp nhìn ra chính là **trong suốt**.

## v5: cấu hình đã chốt thành **mặc định**

```
dieu_khien = 'densepose'        (thay 'dwpose')
ngan_sach_pixel = 458752        → 512×896 (thay 331776 → 432×768)
```

v4 vẫn mặc định `dwpose @ 432×768`, tức gọi mà không truyền gì thì **vẫn ra tay trong suốt** — bẫy im lặng. v5 bỏ bẫy đó.

Bảng đo thật (33 khung, Q5_K_M, lightx2v, seed 42, cùng một cặp đầu vào):

| control | res | giây | trang phục | tay |
|---|---|---|---|---|
| dwpose | 432×768 | 348 | giữ | **trong suốt** |
| densepose | 432×768 | 316 | giữ | **trong suốt** |
| depth | 432×768 | 376 | **mất** | đặc |
| canny | 432×768 | 295 | **mất** | đặc |
| dwpose | 512×896 | 440 | giữ | đặc |
| **densepose** | **512×896** | **399** | **giữ** | **đặc, sạch nhất** |

**Hai quy luật:**

1. **Nền của ảnh điều khiển quyết định model nghe ai.** Nền đen (`dwpose`, `densepose`) → ngoại hình lấy từ ảnh tham chiếu, **giữ trang phục**. Ảnh đầy đủ (`depth`, `canny`) → chép luôn cảnh và quần áo của video mẫu, **mất trang phục**.
2. **Che khuất là chuyện độ phân giải, không phải loại control.** Ở 432 cả hai control nền đen đều trong suốt; lên 512×896 cả hai đều đặc.

Giả thuyết ban đầu *"densepose thắng nhờ khối người đặc"* **sai** — nó chỉ thắng ở 512, và thắng nhờ **nhanh hơn `dwpose` 41 giây**.

**Đã bác bỏ, đừng thử lại:** `pose_depth` (xương bị vẽ thẳng vào khung hình như vật thể), `canny` (mang theo viền hậu cảnh video mẫu), tăng `steps` (`depth` làm tay đặc ở đúng 4 bước).

**Chưa kiểm:** mọi phép đo trên dùng video mẫu `70s-game-show-host` — chuyển động **yếu** (hạng 15/22 trong bộ input, chênh khung liên tiếp 4,40). Chuyển động mạnh (múa, xoay người) chưa thử.

## v4: kết quả đo v3 và cần gạt `densepose`

Ba job v3 (348 / 376 / 352 giây, 33 khung 432×768) cho một quy luật sạch:

| `dieu_khien` | ngoại hình theo ai | che khuất |
|---|---|---|
| `dwpose` — xương trên **nền đen** | **ảnh tham chiếu** → giữ trang phục | ❌ tay trong suốt |
| `depth` — một **ảnh đầy đủ** | **video mẫu** → mất trang phục | ✅ tay đặc |
| `pose_depth` — xương chồng lên depth | hỏng cả hai | — |

**Nền của ảnh điều khiển quyết định model nghe ai.** Nền đen thì nó không có gì để chép ngoài ảnh tham chiếu; nền là ảnh thật thì nó chép luôn cảnh và quần áo của video mẫu.

`pose_depth` là **đường cụt**: khi nền trông như ảnh thật, mấy nét xương bị **vẽ thẳng vào khung hình như vật thể** (thấy rõ xương màu và chấm trên mặt). Ghi lại để không thử lại.

Nên v4 thêm **`densepose`** — thân người tô đặc theo bộ phận, **trên nền đen**: có nền đen như `dwpose` *và* khối đặc như `depth`.

Một thứ đã **loại trừ**: không phải thiếu ngân sách denoise. `depth` làm tay đặc ở đúng 4 bước như `dwpose`, nên tăng `steps` không phải lời giải.

## v3: ba cần gạt, chọn sao cho KHÔNG tăng thời gian sinh

| | Tham số | Chi phí |
|---|---|---|
| E2 | `dieu_khien`: `'depth'` \| `'pose_depth'` | +~8 s/job |
| E3 | `lora`: `'lightx2v'` (mặc định) \| `'causvid'` | **0** — vẫn 4 bước |
| E4 | `QUANT_VACE` = `Q5_K_M` (cell 1) | 0 khi sinh, tải thêm 1,4 GB một lần |

Tải **cả hai LoRA** lúc cài (~0,9 GB) để một Colab đổi qua lại được giữa hai bản chưng cất mà không phải tải lại 13 GB DiT.

**Hai cảnh báo:**
- `depth` **mang theo bố cục nền của video mẫu** — hậu cảnh có thể bị kéo theo cảnh dẫn động. `pose_depth` sinh ra để né chuyện đó, nhưng nó là tổ hợp **ngoài phân bố huấn luyện** của VACE. Cả hai là đường **thử**, không phải mặc định.
- `Q5_K_M` (13,0 GB) chừa ~2,4 GB VRAM cho kích hoạt trên T4 — chạy được nhưng không rộng. OOM thì hạ một dòng về `Q4_K_M`.

**`negative_prompt` ở notebook này KHÔNG có tác dụng** và không sửa được: cfg=1 làm ComfyUI bỏ hẳn nhánh uncond, mà CausVid/lightx2v là LoRA cfg-step-distill nên nâng cfg là phá chưng cất. Đừng tốn job test nó.

Dung lượng thật (đo bằng `content-length`): Q4_K_M 11,6 GB · Q5_0 12,5 · Q5_K_M 13,0 · Q6_K 14,5 · **Q8_0 18,7 — KHÔNG vừa T4** dù tác giả gốc để mặc định.

Lặp lại đúng v2 để so: `QUANT_VACE='Q4_K_M'`, `lora='causvid'`, `dieu_khien='dwpose'`.

**v3 CHƯA đo lại.**

Cell cuối giữ chạy; đợi `PUBLIC_URL=` rồi `GET /version`.

In [ ]:
# ============================================================================
#  ImageVideoToVideo_WanVACE_Colab.ipynb
#  backend = wan_vace_dk   (1 notebook = 1 phuong an, KHONG gop nhieu model 1 file)
#  Port tu Wan2_1_VACE_&_CausVid_LoRA_4_Image+ControlVideo_to_Video_1.ipynb  (repo https://github.com/Isi-dev/Google-Colab_Notebooks @ ae58404)
# ============================================================================
NOTEBOOK_NAME    = 'ImageVideoToVideo_WanVACE_Colab.ipynb'
NOTEBOOK_VERSION = 'v5'        # <-- BUMP moi khi sua notebook + cap nhat CHANGELOG
TEST_CASE        = 'V5-41f-8fps-512x896-Q5KM-lightx2v-densepose'    #@param {type:"string"}
BACKEND          = 'wan_vace_dk'

# Quant cua DiT — lua chon LUC CAI DAT, doi la phai tai lai nen KHONG phai tham so job.
# Do that bang content-length: Q4_K_M 11,6 GB | Q5_0 12,5 | Q5_K_M 13,0 | Q6_K 14,5 | Q8_0 18,7.
# T4 co 15,4 GB VRAM dung duoc: Q5_K_M con ~2,4 GB cho kich hoat — chay duoc nhung khong rong.
# OOM thi ha ve Q4_K_M (day la ban v2 da chay that, 339 s/job).
# Q8_0 18,7 GB KHONG vua T4 du tac gia goc de mac dinh — dung theo.
QUANT_VACE = 'Q5_K_M'    #@param ["Q4_K_M", "Q5_0", "Q5_K_M", "Q6_K"]

# Tên type trong ai_types của liveportrait-auto/config.json. api dựng đường dẫn
# THẲNG từ tên này ('/generate/<type>'), nên nó phải khớp từng ký tự.
AI_TYPE = 'wan_vace_control_video'
AI_TYPE_ALIAS = ['vace_i2v', 'wan_vace', 'video_dieu_khien']

# Cong bo qua GET /version -> khong bao gio nham ban nao / model nao / case nao
MODELS = [
    {
        "name": "Wan2.1 VACE 14B GGUF (quant chon duoc) + CausVid HOAC lightx2v LoRA + dwpose/depth + RIFE 4.25",
        "code": "https://github.com/ali-vilab/VACE",
        "weights": "hf:Isi99999/Wan2.1BasedModels + umt5-xxl fp8 + wan_2.1_vae + CA HAI LoRA (CausVid rank32, lightx2v v2 rank32) de doi duoc giua hai job; dwpose, Depth Anything V2 va DensePose r50 tu tai o lan dung dau (~250 MB); RIFE flownet 4.25",
        "license": "Apache-2.0 (Wan/VACE) — CausVid / lightx2v LoRA: xem giay phep goc",
        "size": "DO THAT bang content-length: Q4_K_M 11,6 GB | Q5_0 12,5 | Q5_K_M 13,0 | Q6_K 14,5 | Q8_0 18,7. Cong umt5 6,7 GB + VAE + 2 LoRA (~0,9 GB) + dwpose ~300 MB + depth vitl ~1,3 GB."
    }
]

BACKEND_PARAMS = ('anh_url', 'video_url', 'chuyen_dong', 'negative_prompt', 'so_frame', 'moi_khung_thu', 'khung_moi_giay', 'rong', 'cao', 'steps', 'guidance', 'seed', 'fps_ra', 'khop_mau', 'dieu_khien', 'lora', 'lora_cuong_do')

# Bộ mặc định = ĐÚNG bộ tác giả gốc đã đo trên T4 (xem cell 0). Đổi là phải đo lại.
MAC_DINH = {
    "so_frame": 41,
    "moi_khung_thu": 0,
    "khung_moi_giay": 8.0,
    "fps_ra": 24,
    "dwpose_gpu": True,
    # 458752 = 512x896. DOI TU 331776 (432x768) o v5 sau khi do duoc: o 432 be ngang
    # ban tay chi ~70 px va model KHONG dung duoc no thanh khoi dac — chu tren bien
    # quang cao lot xuyen qua long ban tay voi CA dwpose LAN densepose. Len 512x896
    # (+38% pixel) thi ca hai deu dac. Gia: +83 giay moi job (316 -> 399 s).
    "ngan_sach_pixel": 458752,
    "steps": 4,
    "guidance": 1.0,
    "sampler": "euler_ancestral",
    "scheduler": "simple",
    "shift": 8.0,
    "causvid": 0.8,
    # Ban chung cat mac dinh. 'causvid' = dung y ban v2 (de so sanh nguoc).
    # 'lightx2v' = ban chung cat doi sau, van 4 buoc nen KHONG cham hon.
    "lora": "lightx2v",
    "lightx2v": 1.0,
    # Depth Anything V2. vitl ~1,3 GB, ~0,2 s/khung tren T4 => ~8 s cho 41 khung,
    # khong dang ke so voi ~300 s denoise. vits (~99 MB) nhanh hon nhung tho hon.
    "depth_ckpt": "depth_anything_v2_vitl.pth",
    "depth_res": 512,
    # DensePose: ban do THAN NGUOI TO DAC theo bo phan, tren NEN DEN. r50 ~250 MB.
    # cmap viridis la bang mau MagicAnimate dung de dieu khien hoat hoa.
    "densepose_model": "densepose_r50_fpn_dl.torchscript",
    "densepose_cmap": "Viridis (MagicAnimate)",
    "densepose_res": 512,
    # Nguong coi mot diem la NET xuong khi chong pose len depth (0..1).
    "pose_nguong": 0.08,
    "khop_mau": True,
    "chuyen_dong": "A person moving naturally, realistic motion, photorealistic, consistent identity, face and clothing kept exactly as in the reference image, high quality, stable camera.",
    "negative_prompt": "bad quality, blurry, messy, chaotic, deformed, extra limbs"
}

CHANGELOG = {
    "v5": "Dua cau hinh DA CHOT thanh mac dinh: dieu_khien='densepose' va ngan_sach_pixel "
          "458752 (512x896, thay 431776/432x768). Ly do: v4 van mac dinh dwpose @ 432x768 nen "
          "goi ma khong truyen gi thi VAN ra tay trong suot — bay im lang cho nguoi dung sau. "
          "BANG DO THAT 16/09/2026 (33 khung, Q5_K_M, lightx2v, seed 42, cung mot cap dau vao): "
          "dwpose 432x768 348s giu-ao/TRONG SUOT | densepose 432x768 316s giu-ao/TRONG SUOT | "
          "depth 432x768 376s MAT-AO/dac | canny 432x768 295s MAT-AO/dac | dwpose 512x896 440s "
          "giu-ao/dac | densepose 512x896 399s giu-ao/dac-sach-nhat <- CHOT. "
          "HAI QUY LUAT rut ra: (a) NEN cua anh dieu khien quyet dinh model nghe ai — nen den "
          "(dwpose, densepose) thi ngoai hinh lay tu anh tham chieu nen GIU TRANG PHUC; anh day "
          "du (depth, canny) thi model chep luon canh va quan ao cua video mau nen MAT TRANG PHUC. "
          "(b) Che khuat la chuyen DO PHAN GIAI chu khong phai loai control — o 432 ca dwpose lan "
          "densepose deu trong suot, len 512x896 ca hai deu dac. "
          "Gia thuyet ban dau cua toi ('densepose thang nho khoi nguoi dac') SAI: no chi thang o "
          "512 va thang nho NHANH HON dwpose 41 giay, khong phai nho khoi dac. "
          "DA BAC BO, dung thu lai: pose_depth (xuong bi VE THANG vao khung hinh nhu vat the vi "
          "nen trong nhu anh that), canny (mang theo vien hau canh video mau), va tang steps "
          "(depth lam tay dac o DUNG 4 buoc, nen khong phai thieu ngan sach denoise). "
          "CHUA KIEM: moi phep do tren dung video mau `70s-game-show-host` — chuyen dong YEU "
          "(xep hang 15/22 trong bo input, chenh khung lien tiep 4,40). Chuyen dong manh (mua, "
          "xoay nguoi) CHUA THU. "
          "CHUA DO LAI v5 (mac dinh moi = dung cau hinh job V4-j2 da do 399 s).",
    "v4": "Them dieu_khien='densepose'. DO THAT ba job v3 tren T4 ngay 16/09/2026 "
          "(348 / 376 / 352 giay, 33 khung 432x768, Q5_K_M, lightx2v) cho mot ket qua "
          "sach va bat ngo: NEN cua anh dieu khien quyet dinh model nghe ai. "
          "(a) dwpose = xuong tren NEN DEN -> giu NGUYEN trang phuc cua anh tham chieu "
          "(non cao boi, ao gi-le tua rua, khan do, sa mac, bien hieu) nhung TAY VAN TRONG SUOT. "
          "(b) depth = mot buc ANH DAY DU -> tay DAC that, nhung model chep luon canh va quan ao "
          "cua video mau: chi con khuon mat cao boi, con lai la ao vest xanh, micro vang va "
          "san khau cua MC. MAT TRANG PHUC. (c) pose_depth = xuong chong len depth -> HONG hon ca "
          "hai: may net xuong BI VE THANG VAO KHUNG HINH nhu vat the that (thay ro xuong mau va "
          "cham tren mat), vi khi nen trong nhu anh that thi model doc net ve la vat the chu khong "
          "phai chi dan. DUONG CUT, da ghi lai de khong thu lai. "
          "Ket luan: can control VUA co nen den (giu trang phuc) VUA co khoi nguoi dac (chua che "
          "khuat) = DensePose. Da co san trong comfyui_controlnet_aux dang cai, r50 torchscript "
          "~250 MB tu tai. Thoi gian: trich dwpose 21,6s / depth ~22s (lan dau 68,8s vi tai model) "
          "/ pose_depth 43,2s; denoise on dinh ~240-250s bat ke control nao, nen doi control gan nhu "
          "KHONG doi tong thoi gian. "
          "Cung ghi lai mot thu da loai tru: khong phai thieu ngan sach denoise. depth lam tay dac "
          "o DUNG 4 buoc nhu dwpose, nen tang steps khong phai loi giai. "
          "CHUA DO LAI v4.",
    "v3": "Ba can gat de do lo TAY TRONG SUOT, chon sao cho KHONG tang thoi gian sinh. "
          "TRIEU CHUNG do lai tren vace-v2-j1: chu tren bang 'WELCOME TO / COSMIC GULCH' hien "
          "XUYEN QUA long ban tay; cang tay gan nhu mat. Da loai tru RIFE: ba ban (v1 4fps RIFE x6, "
          "v1 8fps RIFE x3, v2 8fps) trong suot nhu nhau => loi nam trong latent model sinh ra, "
          "khong phai khau noi suy. CO CHE: doc nodes_wan.py, control_masks=None => mask toan 1 => "
          "inactive = xam tron, reactive = dung bo xuong dwpose; tuc TOAN khung hinh la vung phai "
          "sinh moi, va tin hieu duy nhat noi 'cho nay co canh tay' la bo xuong que tren nen den. "
          "Khong silhouette, khong depth, khong alpha. O 4 buoc + cfg 1 + Q4 model khong giai duoc "
          "mau thuan 'tay o day' vs 'bang quang cao o day' nen lay trung binh — trung binh = trong suot. "
          "(1) E2 dieu_khien them 'depth' (Depth Anything V2, ~0,2 s/khung ~ 8 s/job) va 'pose_depth' "
          "(xuong chong len depth); depth la control type VACE duoc huan luyen chinh thuc. Canh bao: "
          "'depth' mang theo BO CUC NEN cua video mau nen hau canh co the bi keo theo. "
          "(2) E3 tham so `lora`: doi CausVid <-> lightx2v v2 rank32 ngay trong job, van 4 buoc nen "
          "khong cham hon; TAI CA HAI luc cai (~0,9 GB) de mot Colab so sanh duoc ca hai. "
          "(3) E4 QUANT_VACE mac dinh Q5_K_M (13,0 GB) thay Q4_K_M (11,6 GB) — chi tai lau them "
          "~1,4 GB, thoi gian sinh khong doi. T4 con ~2,4 GB cho kich hoat: OOM thi ha ve Q4_K_M. "
          "Sua so lieu sai tu v1: MODELS ghi Q4_K_M '~9 GB', do content-length that la 11,6 GB. "
          "Ghi nhan Q8_0 = 18,7 GB, KHONG vua T4 du tac gia goc de mac dinh — dung theo. "
          "CHUA DO LAI: tat ca con so tren la suy tu dung luong file va do cua v2, phai chay that. "
          "De lap lai dung v2: lora='causvid', dieu_khien='dwpose', QUANT_VACE='Q4_K_M'.",
    "v2": "DO THAT v1 tren T4 ngay 15/09/2026, ba job: 319 / 308 / 288 s GPU, RAM dinh 2,9 GB, ghep cheo (anh cao boi + dong tac nguoi dan chuong trinh) giu mat va trang phuc, lam dung dong tac. Hai loi thay duoc: (a) TAY VAY BI MO/MA — do net vung tay tut giua hai khung goc roi hoi lai dung khung goc (478 -> 411 -> 470), tuc RIFE x6 noi suy tay di qua xa giua hai khung 4 fps; (b) dwpose ton 86 s cho 21 khung = 4 s/khung = dang chay CPU (mac dinh .onnx). v2 sua bon thu: (1) dwpose torchscript (yolox_l.torchscript.pt + dw-ll_ucoco_384_bs5.torchscript.pt) chay tren CUDA, hong thi lui ve onnx; (2) cache prompt embeds theo (prompt, negative) nhu LTX v6 — bo 30 s/job khi prompt khong doi; (3) mac dinh 8 khung/giay (lay moi khung thu 3 cua video 24 fps, RIFE x3) thay vi 4 — denoise ton hon (~9 latent thay vi 6) nhung tay vay it ma; app doi lai duoc bang khung_moi_giay=4; (4) cat so khung xuong dung so khung co that trong video mau (v1 xin 21 ma video chi co 18 -> 3 khung dieu khien dem xam; do lai thay duoi clip KHONG troi nhung van la denoise vo ich). UOC v2 ~270 s/job cho 4 giay — PHAI DO LAI.",
    "v1": "Port tu Isi-dev Wan2_1_VACE_&_CausVid_LoRA_4_Image+ControlVideo_to_Video vao hop dong cua cum. Khac hai notebook kia: model KHONG bia chuyen dong ma lay khung xuong (dwpose) tu tung khung cua VIDEO MAU roi ve nguoi trong ANH theo dung tu the do — chuyen dong la cua nguoi that. GIU NGUYEN: CausVid 0.8, 4 buoc, cfg 1, euler_ancestral/simple, shift 8, dwpose, khop mau ve anh tham chieu, RIFE noi suy. Tac gia ghi 4 giay < 5 phut tren T4 voi Q8_0 — ban nay chon Q4_K_M (tac gia ghi 'crash thi ha quant'; T4 RAM 12,7 GB) nen PHAI DO LAI. Khac tac gia: (1) sua loi tac gia dung bien toan cuc select_every_nth_frame thay vi tham so, (2) fps thoi gian THAT: lay moi khung thu n -> fps_sinh = fps_mau/n, RIFE nhan dung boi de ve fps_ra, khong tua nhanh nhu ban goc (luu 15 fps roi x6 -> 30 fps), (3) RIFE hong thi lui ve ffmpeg minterpolate, (4) so khung 4n+1, (5) chan video den/NaN, (6) bo depth (them 1,3 GB model) — chi dwpose | canny."
}

import shutil, subprocess


def _sh(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True).stdout.strip()
    except Exception:
        return ''


print(f'=== {NOTEBOOK_NAME} {NOTEBOOK_VERSION} | backend={BACKEND} | case={TEST_CASE} ===')
for m in MODELS:
    print(f"    model: {m['name']}  [{m['license']}]  {m['weights']}")
print('GPU:', _sh(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'])
      or '!!! CHUA BAT GPU: Runtime > Change runtime type > GPU')
print('GPU yeu cau:', 'T4 (16 GB) — tac gia goc do: 4s < 5 phut (dwpose, Q8_0); ban nay chon Q4_K_M')
print('ffmpeg:', shutil.which('ffmpeg') or 'THIEU (cell cai dat se cai)')


=== ImageVideoToVideo_WanVACE_Colab.ipynb v5 | backend=wan_vace_dk | case=V5-41f-8fps-512x896-Q5KM-lightx2v-densepose ===
    model: Wan2.1 VACE 14B GGUF (quant chon duoc) + CausVid HOAC lightx2v LoRA + dwpose/depth + RIFE 4.25  [Apache-2.0 (Wan/VACE) — CausVid / lightx2v LoRA: xem giay phep goc]  hf:Isi99999/Wan2.1BasedModels + umt5-xxl fp8 + wan_2.1_vae + CA HAI LoRA (CausVid rank32, lightx2v v2 rank32) de doi duoc giua hai job; dwpose, Depth Anything V2 va DensePose r50 tu tai o lan dung dau (~250 MB); RIFE flownet 4.25
GPU: Tesla T4, 15360 MiB
GPU yeu cau: T4 (16 GB) — tac gia goc do: 4s < 5 phut (dwpose, Q8_0); ban nay chon Q4_K_M
ffmpeg: /usr/bin/ffmpeg


In [ ]:
import os, shutil, subprocess, sys, time

COMFY = '/content/ComfyUI'
_T0 = time.time()


def _run(cmd, cwd=None, check=True, quiet=True):
    """Chay lenh, in dong lenh truoc de log Colab doc duoc dang o buoc nao."""
    print('$', ' '.join(cmd) if isinstance(cmd, list) else cmd, flush=True)
    r = subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str),
                       capture_output=quiet, text=True)
    if check and r.returncode != 0:
        print(r.stdout[-3000:] if r.stdout else '')
        print(r.stderr[-3000:] if r.stderr else '')
        # Dua duoi stderr VAO thong diep loi: traceback Colab hien no ngay, khong phai
        # cuon len tim (lan dau vo, chi thay 'lenh loi (2)' ma khong biet vi sao).
        duoi = (r.stderr or r.stdout or '').strip()[-1200:]
        raise RuntimeError(f'lenh loi ({r.returncode}): {cmd}' + chr(10) + '--- stderr ---' + chr(10) + duoi)
    return r


def _pip(*pk):
    _run([sys.executable, '-m', 'pip', 'install', '-q', *pk])


def _clone(url, dst, branch=None):
    if os.path.isdir(os.path.join(dst, '.git')):
        print(f'   da co {dst}, bo qua clone')
        return
    cmd = ['git', 'clone', '-q', '--depth', '1']
    if branch:
        cmd += ['--branch', branch]
    _run(cmd + [url, dst])


def _aria(url, thu_muc, ten=None):
    """Tai bang aria2c 16 luong (cach tac gia goc). Co roi thi bo qua."""
    ten = ten or url.split('/')[-1].split('?')[0]
    os.makedirs(thu_muc, exist_ok=True)
    d = os.path.join(thu_muc, ten)
    if os.path.exists(d) and os.path.getsize(d) > 1e6:
        print(f'   da co {ten} ({os.path.getsize(d) / 1e9:.2f} GB)')
        return d
    t0 = time.time()
    _run(['aria2c', '--console-log-level=error', '-c', '-x', '16', '-s', '16', '-k', '1M',
          '--summary-interval=0', '-d', thu_muc, '-o', ten, url])
    print(f'   {ten}: {os.path.getsize(d) / 1e9:.2f} GB trong {time.time() - t0:.0f}s')
    return d


_run(['apt-get', '-y', 'install', '-qq', 'aria2', 'ffmpeg'], check=False)

# cloudflared — cell server goi thang /usr/local/bin/cloudflared va KIEM LAI ngay sau khi
# cai (bai hoc tu notebook Qwen: cai xong, model san sang, roi vo o dong cuoi cung).
if not shutil.which('cloudflared'):
    _run('curl -sL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/'
         'releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared')
if not shutil.which('cloudflared'):
    raise RuntimeError('khong cai duoc cloudflared -- cell server se khong mo duoc tunnel')

# Torch GHIM dung ban tac gia goc dung. Colab doi torch moi lien tuc; ban khac la
# xformers/sageattention khong khop wheel. Cai lai mat vai phut nhung chac.
_pip('torch==2.6.0', 'torchvision==0.21.0', 'torchaudio==2.6.0')

_pip('torchsde', 'av', 'diffusers', 'transformers', 'xformers==0.0.29.post2', 'accelerate',
     'tqdm', 'color-matcher', 'onnxruntime', 'onnxruntime-gpu', 'einops', 'spandrel',
     'opencv-python', 'imageio[ffmpeg]', 'psutil', 'fastapi', 'uvicorn', 'python-multipart',
     'requests')

_clone('https://github.com/Isi-dev/ComfyUI', COMFY, branch='ComfyUI_v0.3.36')
_clone('https://github.com/Isi-dev/ComfyUI_GGUF.git', f'{COMFY}/custom_nodes/ComfyUI_GGUF')
_clone('https://github.com/Isi-dev/comfyui_controlnet_aux', f'{COMFY}/custom_nodes/comfyui_controlnet_aux')
# KHONG cai ComfyUI/requirements.txt (xem ly do o notebook Wan22): chi bo duoi cho `nodes`.
_pip('aiohttp', 'yarl', 'pyyaml', 'scipy', 'tqdm', 'psutil', 'safetensors', 'sentencepiece',
     'tokenizers', 'kornia', 'soundfile', 'pydantic', 'pydantic-settings')
_pip('-r', f'{COMFY}/custom_nodes/ComfyUI_GGUF/requirements.txt')
_pip('-r', f'{COMFY}/custom_nodes/comfyui_controlnet_aux/requirements.txt')

# RIFE: noi suy khung. Model chi sinh ~4 khung/giay (lay moi khung thu 6 cua video mau)
# roi RIFE nhan len 24 fps. Khong co RIFE thi video giat.
RIFE = '/content/Practical-RIFE'
_clone('https://github.com/Isi-dev/Practical-RIFE', RIFE)
_pip('git+https://github.com/rk-exxec/scikit-video.git@numpy_deprecation')
os.makedirs(f'{RIFE}/train_log', exist_ok=True)
for _f in ('IFNet_HDv3.py', 'RIFE_HDv3.py', 'refine.py', 'flownet.pkl'):
    if not os.path.exists(f'{RIFE}/train_log/{_f}'):
        _run(f'curl -sL -o {RIFE}/train_log/{_f} https://huggingface.co/Isi99999/'
             f'Frame_Interpolation_Models/resolve/main/4.25/train_log/{_f}')

# Quant lay tu QUANT_VACE o cell 1. Repo goc dat ten file theo HAI kieu khac nhau
# tuy quant — viet thang ra bang chu khong ghep chuoi, vi ghep chuoi thi Q5_K_M se
# thanh 'Wan2.1_14B_VACE-Q5_K_M.gguf' (khong ton tai) va aria2 bao 404 sau vai phut.
QUANT = QUANT_VACE
_TEN_GGUF = {
    'Q4_K_M': 'Wan2.1_14B_VACE-Q4_K_M.gguf',
    'Q5_0':   'Wan2.1_14B_VACE-Q5_0.gguf',
    'Q5_K_M': 'Wan2.1-VACE-14B-Q5_K_M.gguf',
    'Q6_K':   'Wan2.1-VACE-14B-Q6_K.gguf',
}
if QUANT not in _TEN_GGUF:
    raise RuntimeError(f'QUANT_VACE={QUANT!r} khong co trong bang; chon mot trong {list(_TEN_GGUF)}')
DM = f'{COMFY}/models/diffusion_models'
DIT = _aria(f'https://huggingface.co/Isi99999/Wan2.1BasedModels/resolve/main/{_TEN_GGUF[QUANT]}', DM)
# TAI CA HAI LoRA (~0,9 GB tong). Cung mot Colab doi duoc giua hai job bang tham so
# `lora`, khong phai khoi dong lai va tai lai 13 GB DiT chi de so sanh hai ban chung cat.
LORA_CAUSVID = _aria('https://huggingface.co/Isi99999/Wan2.1BasedModels/resolve/main/Wan21_CausVid_14B_T2V_lora_rank32.safetensors', f'{COMFY}/models/loras')
LORA_LIGHTX2V = _aria('https://huggingface.co/Isi99999/Wan2.1BasedModels/resolve/main/lightx2v_T2V_14B_cfg_step_distill_v2_lora_rank32_bf16.safetensors', f'{COMFY}/models/loras')
TE = _aria('https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors', f'{COMFY}/models/text_encoders')
VAE = _aria('https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors', f'{COMFY}/models/vae')
# dwpose (yolox + dw-ll) do comfyui_controlnet_aux tu tai tu HF o lan dung dau (~300 MB).

print(f'>>> CAI DAT XONG ({time.time() - _T0:.0f}s).')


$ apt-get -y install -qq aria2 ffmpeg
$ curl -sL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
$ /usr/bin/python3 -m pip install -q torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0
$ /usr/bin/python3 -m pip install -q torchsde av diffusers transformers xformers==0.0.29.post2 accelerate tqdm color-matcher onnxruntime onnxruntime-gpu einops spandrel opencv-python imageio[ffmpeg] psutil fastapi uvicorn python-multipart requests
$ git clone -q --depth 1 --branch ComfyUI_v0.3.36 https://github.com/Isi-dev/ComfyUI /content/ComfyUI
$ git clone -q --depth 1 https://github.com/Isi-dev/ComfyUI_GGUF.git /content/ComfyUI/custom_nodes/ComfyUI_GGUF
$ git clone -q --depth 1 https://github.com/Isi-dev/comfyui_controlnet_aux /content/ComfyUI/custom_nodes/comfyui_controlnet_aux
$ /usr/bin/python3 -m pip install -q aiohttp yarl pyyaml scipy tqdm psutil safetensors sentencepiece tokenizers k

In [ ]:
import glob, json, os, re, shutil, subprocess, time
from typing import Optional

import numpy as np
from PIL import Image, ImageOps
from pydantic import BaseModel

JOB_DIR = '/content/jobs'
RES_DIR = '/content/results'
WRK_DIR = '/content/work'
for d in (JOB_DIR, RES_DIR, WRK_DIR):
    os.makedirs(d, exist_ok=True)

LAST_STATS = {}


def _slug(s, default='case'):
    s = re.sub(r'[^A-Za-z0-9._-]+', '-', (s or '').strip()).strip('-.')
    return (s or default)[:48]


def _moc(ten):
    """Moc bo nho (RAM/VRAM) tai tung buoc. T4 hay chet o RAM 12,7 GB chu khong phai
    VRAM, nen phai thay duoc RAM tang o buoc nao."""
    try:
        import psutil
        ram = round(psutil.Process(os.getpid()).memory_info().rss / 1024 ** 3, 2)
    except Exception:
        ram = None
    try:
        import torch
        vram = round(torch.cuda.memory_allocated() / 1024 ** 3, 2) if torch.cuda.is_available() else None
    except Exception:
        vram = None
    LAST_STATS.setdefault('_bo_nho', {})[ten] = {'ram_gb': ram, 'vram_gb': vram, 't': round(time.time(), 1)}


def probe_video(path):
    out = subprocess.run(
        ['ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries',
         'stream=width,height,r_frame_rate,nb_frames', '-of', 'json', path],
        capture_output=True, text=True, check=True).stdout
    st = json.loads(out)['streams'][0]
    num, den = (st['r_frame_rate'].split('/') + ['1'])[:2]
    fps = float(num) / float(den or 1)
    return int(st['width']), int(st['height']), fps, int(st.get('nb_frames') or 0)


def encode_video(frames_dir, fps, out_path, tag=''):
    """PNG -> mp4 h264 yuv420p. Metadata comment mang du notebook + version + backend + case
    -> ffprobe file mp4 la biet ngay ban nao sinh ra no."""
    comment = (f'notebook={NOTEBOOK_NAME} version={NOTEBOOK_VERSION} backend={BACKEND} {tag}').strip()
    subprocess.run(['ffmpeg', '-v', 'error', '-y', '-framerate', f'{fps:.6f}', '-start_number', '0',
                    '-i', os.path.join(frames_dir, '%06d.png'),
                    '-c:v', 'libx264', '-preset', 'medium', '-crf', '18', '-pix_fmt', 'yuv420p',
                    '-metadata', f'comment={comment}', out_path], check=True)
    return out_path


def reencode_video(src, out_path, tag=''):
    comment = (f'notebook={NOTEBOOK_NAME} version={NOTEBOOK_VERSION} backend={BACKEND} {tag}').strip()
    subprocess.run(['ffmpeg', '-v', 'error', '-y', '-i', src, '-c:v', 'libx264', '-preset', 'medium',
                    '-crf', '18', '-pix_fmt', 'yuv420p', '-metadata', f'comment={comment}', out_path],
                   check=True)
    return out_path


def save_frames_u8(arr, out_dir):
    """arr: (N,H,W,3) uint8 -> 000000.png ..."""
    if os.path.isdir(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(out_dir, exist_ok=True)
    for i in range(arr.shape[0]):
        Image.fromarray(arr[i]).save(os.path.join(out_dir, f'{i:06d}.png'))
    return out_dir


def doc_anh_rgb(path):
    """PNG RGBA (khuon mat / outfit trong repo la RGBA) -> ep len nen TRANG roi RGB.
    .convert('RGB') thang se bien vung trong suot thanh mau rac."""
    im = Image.open(path)
    im = ImageOps.exif_transpose(im)
    if im.mode in ('RGBA', 'LA') or (im.mode == 'P' and 'transparency' in im.info):
        im = im.convert('RGBA')
        bg = Image.new('RGBA', im.size, (255, 255, 255, 255))
        im = Image.alpha_composite(bg, im)
    return im.convert('RGB')


def _kich_thuoc_theo_anh(w0, h0, ngan_sach, boi=16, toi_thieu=256):
    """Giu ti le anh vao, tong pixel ~= ngan_sach, moi canh chia het cho `boi`.
    Bai hoc LTX v4: cat giua ve 704x512 lam anh doc mat 51% chieu cao (chi con khuon mat)."""
    ti_le = w0 / max(1, h0)
    h = (ngan_sach / ti_le) ** 0.5
    w = h * ti_le
    w = max(toi_thieu, int(round(w / boi)) * boi)
    h = max(toi_thieu, int(round(h / boi)) * boi)
    return w, h


def _khung_4n1(n, toi_thieu=5):
    """Wan: so khung phai la 4n+1. Lam tron XUONG va bao ro."""
    n = max(toi_thieu, int(n))
    return (n - 1) // 4 * 4 + 1


def _khung_8n1(n, toi_thieu=9):
    """LTX: so khung phai la 8n+1."""
    n = max(toi_thieu, int(n))
    return (n - 1) // 8 * 8 + 1


def _tai_ve(url, dst):
    from urllib.parse import urlsplit
    import requests
    sp = urlsplit(url)
    r = requests.get(url, stream=True, timeout=300,
                     headers={'User-Agent': 'Mozilla/5.0', 'Referer': f'{sp.scheme}://{sp.netloc}/'})
    r.raise_for_status()
    with open(dst, 'wb') as f:
        for chunk in r.iter_content(1 << 16):
            f.write(chunk)
    return dst


def _duoi(url, mac_dinh):
    from urllib.parse import urlparse
    e = os.path.splitext(urlparse(url or '').path or (url or ''))[1].lower()
    return e if e else mac_dinh


class Job(BaseModel):
    """Than JSON cua POST /generate/<type>.

    anh_url  : anh NGUOI (tham chieu — khuon mat, quan ao lay tu day)
    video_url: video MAU — chi lay CHUYEN DONG (khung xuong dwpose), khong lay hinh
    """
    anh_url: str
    video_url: str
    test_case: str = ''
    chuyen_dong: str = ''
    negative_prompt: str = ''
    # None = de MAC_DINH quyet.
    so_frame: Optional[int] = None          # so khung model SINH (truoc noi suy); tu cat xuong so khung co that
    moi_khung_thu: Optional[int] = None     # lay moi khung thu n cua video mau; 0 = tu tinh theo khung_moi_giay
    khung_moi_giay: Optional[float] = None  # 8 = tay vay it ma; 4 = nhanh hon ~30% nhung RIFE x6 lam ma tay
    rong: Optional[int] = None
    cao: Optional[int] = None
    steps: Optional[int] = None
    guidance: Optional[float] = None
    fps_ra: Optional[int] = None            # fps video cuoi (sau RIFE)
    khop_mau: Optional[bool] = None         # ep mau video ve mau anh tham chieu
    # 'densepose'  : than nguoi TO DAC theo bo phan, tren NEN DEN — to hop cua hai
    #                cai duoi, nham giu trang phuc MA tay van dac (mac dinh tu v4)
    # 'dwpose'     : khung xuong tren nen den (ban v1/v2)
    # 'depth'      : ban do do sau ca khung — CO tin hieu che khuat, nhung MANG THEO
    #                bo cuc nen cua video mau, nen hau canh co the bi keo theo video mau
    # 'pose_depth' : khung xuong chong len depth — giu khop chinh xac cua pose VA the
    #                tich dac cua depth. Day la to hop NGOAI phan bo huan luyen cua VACE.
    # 'canny'      : vien
    dieu_khien: str = 'densepose'
    lora: Optional[str] = None              # 'lightx2v' (mac dinh) | 'causvid' (nhu v2)
    lora_cuong_do: Optional[float] = None
    seed: int = 42
    job_id: Optional[str] = None
    callback_url: Optional[str] = None
    callback_token: Optional[str] = None


def chay_job(job, uid, out_path, progress=None):
    if progress:
        progress('tai_anh', 0, 2)
    goc = _tai_ve(job.anh_url, os.path.join(JOB_DIR, f'{uid}_a{_duoi(job.anh_url, ".png")}'))
    anh = os.path.join(JOB_DIR, f'{uid}_a.png')
    doc_anh_rgb(goc).save(anh)
    if progress:
        progress('tai_video', 1, 2)
    video = _tai_ve(job.video_url, os.path.join(JOB_DIR, f'{uid}_v{_duoi(job.video_url, ".mp4")}'))
    if job.dieu_khien not in ('densepose', 'dwpose', 'canny', 'depth', 'pose_depth'):
        raise ValueError("dieu_khien phai la 'densepose' | 'dwpose' | 'depth' | 'pose_depth' "
                         f"| 'canny', nhan {job.dieu_khien!r}")
    if job.lora is not None and job.lora not in ('causvid', 'lightx2v'):
        raise ValueError(f"lora phai la 'causvid' hoac 'lightx2v', nhan {job.lora!r}")
    # rong/cao KHONG co trong MAC_DINH: None = suy tu ti le anh vao (run_backend lo).
    th = {k: (getattr(job, k) if getattr(job, k) is not None else MAC_DINH.get(k))
          for k in ('so_frame', 'moi_khung_thu', 'khung_moi_giay', 'rong', 'cao', 'steps', 'guidance',
                    'fps_ra', 'khop_mau', 'lora')}
    th['lora_cuong_do'] = job.lora_cuong_do
    return run_backend(anh, video, out_path, test_case=job.test_case or TEST_CASE,
                       chuyen_dong=job.chuyen_dong or MAC_DINH['chuyen_dong'],
                       negative_prompt=job.negative_prompt or MAC_DINH['negative_prompt'],
                       dieu_khien=job.dieu_khien, seed=job.seed, progress=progress, **th)


In [ ]:
import gc, os, subprocess, sys, time
import numpy as np
import torch

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
sys.path.insert(0, COMFY)
os.chdir(COMFY)   # ComfyUI tim models/ input/ theo cwd o vai cho

import comfy.utils
from comfy import model_management

_TIEN_DO = {'cb': None, 'buoc': ''}


def _bat_tien_do(progress):
    """Noi thanh tien do cua ComfyUI (tung buoc denoise) vao progress cua job."""
    _TIEN_DO['cb'] = progress

    def hook(value, total, *a, **k):
        cb = _TIEN_DO['cb']
        if cb and total:
            cb(_TIEN_DO['buoc'], value, total)
    comfy.utils.set_progress_bar_global_hook(hook)


def _buoc(ten, progress, done=0, total=1):
    _TIEN_DO['buoc'] = ten
    if progress:
        progress(ten, done, total)
    _moc(ten)
    print(f'   [{BACKEND}] {ten} ...', flush=True)


def _don():
    try:
        model_management.unload_all_models()
    except Exception:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def _tensor_sang_u8(decoded):
    """(N,H,W,3) float [0,1] -> uint8. Kiem NaN/den o day: ComfyUI KHONG bao, cu tra
    tensor rac roi ta ghi ra video den ma job van 'done' (bai hoc Qwen)."""
    x = decoded.detach().float().cpu()
    if not torch.isfinite(x).all():
        raise RuntimeError('tensor giai ma co NaN/Inf — pipeline tran so (fp16 tren T4?)')
    x = (x.clamp(0, 1) * 255).round().to(torch.uint8).numpy()
    if x.std() < 0.5:
        raise RuntimeError(f'video ra RONG (do lech chuan {x.std():.3f}/255) — moi pixel nhu nhau')
    return x


def _kiem_san_sang():
    """Chay luc preload: import node + kiem file model. Nhanh, khong nap gi len GPU."""
    thieu = [p for p in _FILE_CAN if not os.path.exists(p)]
    if thieu:
        raise RuntimeError('thieu file model: ' + ', '.join(thieu))
    print(f'   [{BACKEND}] du {len(_FILE_CAN)} file model, GPU: '
          f'{torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHONG CO"}')


import cv2
from nodes import (CLIPLoader, CLIPTextEncode, VAEDecode, VAELoader, KSampler, LoadImage,
                   LoraLoaderModelOnly, ImageScale)
from custom_nodes.ComfyUI_GGUF.nodes import UnetLoaderGGUF
from comfy_extras.nodes_model_advanced import ModelSamplingSD3
from comfy_extras.nodes_wan import WanVaceToVideo, TrimVideoLatent
from custom_nodes.comfyui_controlnet_aux.node_wrappers.dwpose import DWPose_Preprocessor
from custom_nodes.comfyui_controlnet_aux.node_wrappers.canny import Canny_Edge_Preprocessor
from custom_nodes.comfyui_controlnet_aux.node_wrappers.depth_anything_v2 import (
    Depth_Anything_V2_Preprocessor)
from custom_nodes.comfyui_controlnet_aux.node_wrappers.densepose import DensePose_Preprocessor

# depth_anything_v2 KHONG nam o day: node tu tai tu HF o lan dung dau. Neu mang Colab
# chet luc do thi job depth hong, job dwpose van chay — chu dich khong phai giu ca hai.
_FILE_CAN = [DIT, LORA_CAUSVID, LORA_LIGHTX2V, TE, VAE,
             os.path.join(RIFE, 'train_log', 'flownet.pkl')]
PRELOAD = [('kiem_san_sang', _kiem_san_sang)]

# ── Cache prompt embeds (bai hoc LTX v6) ──────────────────────────────────────
# Do v1: ma hoa prompt ton 30 s/job (nap umt5 fp8 6,7 GB roi encode). Prompt mac dinh
# co dinh nen ket qua cung co dinh. Chi phu thuoc (prompt, negative). Tra BAN SAO tensor.
_EMBEDS, _EMBEDS_TRAN, _EMBEDS_DEM = {}, 16, {'trung': 0, 'truot': 0}


def _sao_cond(cond):
    return [[t.clone() if hasattr(t, 'clone') else t, dict(d)] for t, d in cond]


def _ma_hoa_prompt(chuyen_dong, negative_prompt):
    khoa = (chuyen_dong, negative_prompt)
    if khoa in _EMBEDS:
        _EMBEDS_DEM['trung'] += 1
        pos, neg = _EMBEDS[khoa]
        print(f'   [{BACKEND}] prompt da co trong kho -> bo qua nap umt5 (kho {len(_EMBEDS)}/{_EMBEDS_TRAN})')
        return _sao_cond(pos), _sao_cond(neg)
    _EMBEDS_DEM['truot'] += 1
    clip = CLIPLoader().load_clip(os.path.basename(TE), 'wan', 'default')[0]
    pos = CLIPTextEncode().encode(clip, chuyen_dong)[0]
    neg = CLIPTextEncode().encode(clip, negative_prompt)[0]
    del clip
    _don()
    if len(_EMBEDS) >= _EMBEDS_TRAN:
        _EMBEDS.pop(next(iter(_EMBEDS)))
    _EMBEDS[khoa] = (pos, neg)
    return _sao_cond(pos), _sao_cond(neg)


def _vao_input(anh, uid):
    d = os.path.join(COMFY, 'input')
    os.makedirs(d, exist_ok=True)
    ten = f'{uid}.png'
    import shutil as _sh
    _sh.copyfile(anh, os.path.join(d, ten))
    return ten


def _dwpose(khung):
    """dwpose tren GPU (torchscript) — do v1: ban .onnx mac dinh roi ve CPU, 4 s/khung.
    Torchscript hong (thieu model, thieu CUDA) thi lui ve onnx, ghi ro vao LAST_STATS."""
    if MAC_DINH.get('dwpose_gpu', True):
        try:
            ra = DWPose_Preprocessor().estimate_pose(
                khung, bbox_detector='yolox_l.torchscript.pt',
                pose_estimator='dw-ll_ucoco_384_bs5.torchscript.pt')['result'][0]
            LAST_STATS['dwpose_thiet_bi'] = 'torchscript-gpu'
            return ra
        except Exception as e:
            print(f'   [{BACKEND}] dwpose torchscript hong ({type(e).__name__}: {str(e)[:160]}) -> onnx')
            LAST_STATS['dwpose_thiet_bi'] = f'onnx (torchscript hong: {type(e).__name__})'
    else:
        LAST_STATS['dwpose_thiet_bi'] = 'onnx'
    return DWPose_Preprocessor().estimate_pose(khung)['result'][0]


def _depth(khung):
    """Ban do do sau (Depth Anything V2) cho ca chuoi khung.

    Vi sao can: voi dieu_khien='dwpose', tin hieu DUY NHAT noi 'cho nay co canh tay' la
    mot bo xuong que tren nen den. Khong co silhouette, khong co alpha, khong co thu tu
    xa/gan. Model phai TU SUY ra tay che bang quang cao. O 4 buoc denoise no khong suy
    duoc nen lay trung binh hai lop — va trung binh hai lop nhin ra la TRONG SUOT.
    Depth noi thang: tay gan hon bang. Depth cung la mot control type VACE duoc huan
    luyen chinh thuc, khong phai meo."""
    ra = Depth_Anything_V2_Preprocessor().execute(
        khung, ckpt_name=MAC_DINH['depth_ckpt'], resolution=int(MAC_DINH['depth_res']))[0]
    _don()
    return ra


def _densepose(khung):
    """Ban do than nguoi TO DAC tren NEN DEN (DensePose).

    Day la to hop rut ra tu ba job v3, khong phai y tuong moi:

      - dwpose cho NEN DEN  -> model khong co gi de chep ngoai anh tham chieu,
        nen TRANG PHUC GIU DUOC. Nhung xuong que khong noi duoc "cho nay co khoi
        thit dac", nen tay van trong suot.
      - depth cho KHOI DAC -> tay dac that. Nhung no la mot buc anh day du, nen
        model chep luon canh va quan ao cua video mau -> MAT TRANG PHUC.

    DensePose co ca hai: nen den (nhu dwpose) + than nguoi to dac theo bo phan
    (nhu depth). Ky vong: giu trang phuc VA tay dac.

    KHONG lam ban 'pose_densepose' (chong xuong len densepose): job v3j3 da cho
    thay khi nen trong nhu mot buc anh thi may net xuong bi VE THANG VAO KHUNG
    HINH nhu vat the that (nhin ro xuong va cham tren mat). Densepose cung la
    anh-nhu-that nen se dinh dung loi do."""
    ra = DensePose_Preprocessor().execute(
        khung, model=MAC_DINH['densepose_model'], cmap=MAC_DINH['densepose_cmap'],
        resolution=int(MAC_DINH['densepose_res']))[0]
    _don()
    return ra


def _chong_pose_len_depth(skel, do_sau):
    """Ve bo xuong dwpose de len tren ban do do sau.

    Duoc cai gi: khop chinh xac cua pose (dwpose bat dung co tay/ngon) CONG the tich
    dac cua depth. Mat cai gi: day la to hop NGOAI phan bo huan luyen cua VACE — no
    quen control 'pose' rieng va 'depth' rieng, khong quen hai cai chong nhau. Vi vay
    day la duong THU, khong phai duong mac dinh."""
    nguong = float(MAC_DINH['pose_nguong'])
    co_net = (skel.amax(dim=-1, keepdim=True) > nguong)
    return torch.where(co_net, skel, do_sau)


def _khung_video(video, moi_thu, so_khung):
    """Doc video mau, lay moi khung thu `moi_thu`, toi da `so_khung` -> tensor (N,H,W,3) [0,1]."""
    cap = cv2.VideoCapture(video)
    fps = cap.get(cv2.CAP_PROP_FPS) or 24.0
    ra, i = [], 0
    while len(ra) < so_khung:
        ok, f = cap.read()
        if not ok:
            break
        if i % moi_thu == 0:
            ra.append(torch.from_numpy(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)).float() / 255.0)
        i += 1
    cap.release()
    if not ra:
        raise ValueError('video mau khong doc duoc khung nao')
    return torch.stack(ra, 0), fps


def _khop_mau(anh_tc, video):
    """Ep phan bo mau cua tung khung ve mau anh tham chieu (color-matcher, phuong phap mkl)."""
    from color_matcher import ColorMatcher
    cm = ColorMatcher()
    ref = anh_tc.squeeze(0).cpu().numpy()
    ra = []
    for i in range(video.shape[0]):
        t = video[i].cpu().numpy()
        ra.append(torch.from_numpy(t + 1.0 * (cm.transfer(src=t, ref=ref, method='mkl') - t)))
    return torch.stack(ra).float().clamp(0, 1)


def _rife(src, dst, boi, fps_ra):
    """Noi suy khung x`boi` bang Practical-RIFE. Hong thi lui ve ffmpeg minterpolate."""
    try:
        subprocess.run([sys.executable, 'inference_video.py', f'--multi={boi}', f'--fps={fps_ra}',
                        f'--video={src}', '--scale=1', f'--output={dst}'],
                       cwd=RIFE, check=True, capture_output=True, text=True, timeout=1800)
        if os.path.exists(dst) and os.path.getsize(dst) > 1000:
            return 'rife'
        raise RuntimeError('RIFE khong ghi file')
    except Exception as e:
        print(f'   [{BACKEND}] RIFE hong ({type(e).__name__}: {str(e)[:200]}) -> dung ffmpeg minterpolate')
        subprocess.run(['ffmpeg', '-v', 'error', '-y', '-i', src,
                        '-vf', f"minterpolate='fps={fps_ra}:mi_mode=mci:mc_mode=aobmc:vsbmc=1'",
                        '-c:v', 'libx264', '-crf', '18', '-pix_fmt', 'yuv420p', dst], check=True)
        return 'minterpolate'


def run_backend(anh, video, out_path, test_case='', so_frame=41, moi_khung_thu=0, khung_moi_giay=None,
                rong=None, cao=None, steps=4, guidance=1.0, seed=42, fps_ra=24, khop_mau=True,
                dieu_khien='dwpose', lora=None, lora_cuong_do=None,
                chuyen_dong='', negative_prompt='', progress=None):
    """Port tu generate_video() cua Wan2_1_VACE_&_CausVid_LoRA_4_Image+ControlVideo_to_Video.

    Y tuong: model KHONG bia chuyen dong — no lay khung xuong (dwpose) tu tung khung
    video mau roi ve nguoi trong anh theo dung tu the do. Chuyen dong la cua NGUOI THAT.
    Cai gia: mau chi sinh ~4 khung/giay (lay moi khung thu 6 cua video 24 fps), roi RIFE
    noi suy len fps_ra. Tac gia do tren T4: 4 giay < 5 phut.
    """
    LAST_STATS.clear()
    t0 = time.time()
    _bat_tien_do(progress)
    uid = os.path.splitext(os.path.basename(out_path))[0].split('_')[0]

    from PIL import Image as _I
    w0, h0 = _I.open(anh).size
    if rong is None or cao is None:
        rong, cao = _kich_thuoc_theo_anh(w0, h0, MAC_DINH['ngan_sach_pixel'], boi=16)
    vw, vh, vfps, vn = probe_video(video)
    kmg = float(khung_moi_giay or MAC_DINH['khung_moi_giay'])
    moi_thu = int(moi_khung_thu) if moi_khung_thu else max(1, int(round(vfps / kmg)))
    # Cat xuong so khung CO THAT: v1 xin 21 ma video chi co 18 -> 3 khung dieu khien dem xam,
    # denoise vo ich. vn = 0 nghia la ffprobe khong biet -> khong cat.
    co_that = (vn - 1) // moi_thu + 1 if vn else 10 ** 9
    n = _khung_4n1(min(int(so_frame), co_that))
    if n < so_frame:
        print(f'   [{BACKEND}] so_frame {so_frame} -> {n} (video mau chi co {co_that} khung khi lay moi khung thu {moi_thu}, va can 4n+1)')
    fps_sinh = vfps / moi_thu                     # fps THAT cua chuoi khung model sinh
    boi = max(1, int(round(fps_ra / fps_sinh)))   # RIFE nhan bao nhieu lan de ve fps_ra
    LAST_STATS.update({'_case': test_case, 'anh_vao': [w0, h0], 'video_mau': [vw, vh, vfps, vn],
                       'video_ra': [rong, cao, n, fps_ra], 'moi_khung_thu': moi_thu,
                       'fps_sinh': round(fps_sinh, 2), 'rife_x': boi, 'steps': steps, 'cfg': guidance,
                       'seed': seed, 'dieu_khien': dieu_khien, 'prompt': chuyen_dong[:200],
                       'quant': QUANT_VACE,
                       'lora': (lora or MAC_DINH['lora']),
                       'lora_cuong_do': (lora_cuong_do if lora_cuong_do is not None else
                                         MAC_DINH['lightx2v' if (lora or MAC_DINH['lora']) == 'lightx2v'
                                                  else 'causvid'])})
    print(f'   [{BACKEND}] anh {w0}x{h0}, video mau {vw}x{vh}@{vfps:.0f}fps x{vn} -> lay moi khung thu '
          f'{moi_thu} ({fps_sinh:.1f} fps), sinh {n} khung {rong}x{cao}, RIFE x{boi} -> {fps_ra} fps')

    with torch.inference_mode():
        _buoc('ma_hoa_prompt', progress)
        pos, neg = _ma_hoa_prompt(chuyen_dong, negative_prompt)
        LAST_STATS['_embeds'] = {'so_muc': len(_EMBEDS), **_EMBEDS_DEM}

        _buoc('doc_video_mau', progress)
        img = LoadImage().load_image(_vao_input(anh, uid))[0]
        img = ImageScale().upscale(img, 'lanczos', rong, cao, 'disabled')[0]
        khung, _ = _khung_video(video, moi_thu, n)
        khung = ImageScale().upscale(khung, 'lanczos', rong, cao, 'disabled')[0]

        _buoc('trich_' + dieu_khien, progress)
        if dieu_khien == 'canny':
            dk = Canny_Edge_Preprocessor().execute(khung, 100, 200, 512)[0]
        elif dieu_khien == 'densepose':
            dk = _densepose(khung)
        elif dieu_khien == 'depth':
            dk = _depth(khung)
        elif dieu_khien == 'pose_depth':
            dk = _chong_pose_len_depth(_dwpose(khung), _depth(khung))
        else:
            dk = _dwpose(khung)
        del khung
        _don()

        _buoc('ma_hoa_vace', progress)
        vae = VAELoader().load_vae(os.path.basename(VAE))[0]
        pos_o, neg_o, latent, trim = WanVaceToVideo().encode(pos, neg, vae, rong, cao, n, 1, 1.0, dk, None, img)
        del dk
        _don()

        _buoc('nap_model', progress)
        model = UnetLoaderGGUF().load_unet(os.path.basename(DIT))[0]
        _ten_lora = (lora or MAC_DINH['lora']).lower()
        _tep_lora = LORA_LIGHTX2V if _ten_lora == 'lightx2v' else LORA_CAUSVID
        _cd_lora = float(lora_cuong_do if lora_cuong_do is not None
                         else MAC_DINH['lightx2v' if _ten_lora == 'lightx2v' else 'causvid'])
        print(f'   [{BACKEND}] LoRA {_ten_lora} cuong do {_cd_lora}')
        model = LoraLoaderModelOnly().load_lora_model_only(
            model, os.path.basename(_tep_lora), _cd_lora)[0]
        model = ModelSamplingSD3().patch(model, MAC_DINH['shift'])[0]

        _buoc('denoise', progress)
        sampled = KSampler().sample(model=model, seed=seed, steps=steps, cfg=guidance,
                                    sampler_name=MAC_DINH['sampler'], scheduler=MAC_DINH['scheduler'],
                                    positive=pos_o, negative=neg_o, latent_image=latent)[0]
        del model
        _don()
        sampled = TrimVideoLatent().op(sampled, trim)[0]   # bo khung tham chieu o dau

        _buoc('giai_ma', progress)
        decoded = VAEDecode().decode(vae, sampled)[0]
        del vae, sampled
        _don()
        if khop_mau:
            _buoc('khop_mau', progress)
            decoded = _khop_mau(img, decoded)

    _buoc('ghi_video', progress)
    u8 = _tensor_sang_u8(decoded)
    del decoded
    tho = os.path.join(WRK_DIR, f'{uid}_tho.mp4')
    encode_video(save_frames_u8(u8, os.path.join(WRK_DIR, uid)), fps_sinh, tho, tag='tho')
    _buoc('noi_suy_khung', progress)
    ns = os.path.join(WRK_DIR, f'{uid}_rife.mp4')
    LAST_STATS['noi_suy'] = _rife(tho, ns, boi, fps_ra)
    reencode_video(ns, out_path, tag=f'case={_slug(test_case)} seed={seed}')
    _don()
    LAST_STATS['_thoi_gian_s'] = round(time.time() - t0, 1)
    _moc('xong')
    print(f'   [{BACKEND}] xong {n} khung (+RIFE) trong {time.time() - t0:.0f}s -> {out_path}')
    return out_path


[custom_nodes.comfyui_controlnet_aux] | INFO -> Using ckpts path: /content/ComfyUI/custom_nodes/comfyui_controlnet_aux/ckpts
[custom_nodes.comfyui_controlnet_aux] | INFO -> Using symlinks: False
[custom_nodes.comfyui_controlnet_aux] | INFO -> Using ort providers: ['CUDAExecutionProvider', 'DirectMLExecutionProvider', 'OpenVINOExecutionProvider', 'ROCMExecutionProvider', 'CPUExecutionProvider', 'CoreMLExecutionProvider']
/content/ComfyUI/custom_nodes/comfyui_controlnet_aux/node_wrappers/dwpose.py:26: UserWarning: DWPose: Onnxruntime not found or doesn't come with acceleration providers, switch to OpenCV with CPU device. DWPose might run very slowly
  warnings.warn("DWPose: Onnxruntime not found or doesn't come with acceleration providers, switch to OpenCV with CPU device. DWPose might run very slowly")


In [ ]:
import os, re, socket, subprocess, threading, time, traceback, uuid
from typing import Optional

import requests
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse, HTMLResponse

PORT = 8000
STATE = {'models_ready': False}
app = FastAPI(title=f'{NOTEBOOK_NAME} [{BACKEND}] {NOTEBOOK_VERSION}')


# ============================================================ job store (async)
# Tra job_id ngay roi render o thread nen -> tranh Cloudflare 524 khi job chay vai chuc phut.
JOBS, JOBS_LOCK, RUN_LOCK, JOB_TTL = {}, threading.Lock(), threading.Lock(), 6 * 3600


def _prune_locked():
    now = time.time()
    for k in [k for k, v in JOBS.items() if now - v.get('created', now) > JOB_TTL]:
        v = JOBS.pop(k, None)
        try:
            if v and v.get('result') and os.path.exists(v['result']):
                os.remove(v['result'])
        except Exception:
            pass


def _new_job(jid=None, callback_url=None, callback_token=None):
    jid = jid or uuid.uuid4().hex
    with JOBS_LOCK:
        _prune_locked()
        JOBS[jid] = {'status': 'processing', 'stage': 'queued', 'progress': 0.0,
                     'result': None, 'filename': None, 'error': None, 'created': time.time(),
                     'callback_url': callback_url, 'callback_token': callback_token,
                     'da_day_ve': None}
    return jid


def _progress_cb(jid):
    def cb(stage, done, total):
        with JOBS_LOCK:
            if jid in JOBS:
                JOBS[jid]['stage'] = stage
                JOBS[jid]['progress'] = round(done / max(1, total), 4)
    return cb


# ── Colab tự đẩy kết quả về api ─────────────────────────────────────────────
# Đo trên chính hạ tầng này: quick tunnel cloudflared cho chiều api-KÉO khoảng
# 0,11 MB/s, chiều Colab-ĐẨY khoảng 8,5 MB/s — chênh khoảng 85 lần.
# Không có callback thì KHÔNG làm gì: api kéo qua GET /jobs/{id}/result như cũ.
def day_ve_server(jid, duong):
    with JOBS_LOCK:
        j = JOBS.get(jid) or {}
        url, token = j.get('callback_url'), j.get('callback_token')
    if not url or not token or not duong or not os.path.exists(duong):
        return None
    mb = os.path.getsize(duong) / 1e6
    t0 = time.perf_counter()
    try:
        with open(duong, 'rb') as f:
            r = requests.post(url, data={'token': token},
                              files={'file': (os.path.basename(duong), f, 'video/mp4')},
                              timeout=(15, 900))
        if r.status_code != 200:
            print(f'[{BACKEND}] day ve server that bai HTTP {r.status_code}: {r.text[:200]}')
            return False
        giay = time.perf_counter() - t0
        print(f'[{BACKEND}] da day {mb:.2f} MB ve server trong {giay:.1f}s '
              f'({mb * 1000 / max(giay, .01):.0f} KB/s)')
        with JOBS_LOCK:
            if jid in JOBS:
                JOBS[jid]['da_day_ve'] = True
        return True
    except Exception as e:
        print(f'[{BACKEND}] day ve server loi: {e}')
        return False


def _run_async(jid, tag, work):
    try:
        with RUN_LOCK:                      # serialize GPU: chi 1 job render 1 luc
            path, filename = work()
        with JOBS_LOCK:
            if jid in JOBS:
                JOBS[jid].update(status='done', stage='done', progress=1.0,
                                 result=path, filename=filename,
                                 so_lieu=dict(LAST_STATS))
        day_ve_server(jid, path)
        print(f'[{tag}] xong job {jid[:8]} -> {path}')
    except Exception as e:
        traceback.print_exc()
        tb = traceback.format_exc()
        frames = [ln for ln in tb.splitlines() if ln.strip().startswith('File "')]
        with JOBS_LOCK:
            if jid in JOBS:
                JOBS[jid].update(status='error', stage='error', error=str(e)[:800],
                                 error_type=type(e).__name__,
                                 where=frames[-1].strip()[:300] if frames else None,
                                 traceback=tb[-2500:])
        print(f'[{tag}] LOI job {jid[:8]}: {type(e).__name__}: {e}')


def _spawn(jid, tag, work, test_case=None):
    with JOBS_LOCK:
        if jid in JOBS:
            JOBS[jid]['test_case'] = _slug(test_case or TEST_CASE)
    print(f'[{tag}] nhan job {jid[:8]} case={_slug(test_case or TEST_CASE)} '
          f'({NOTEBOOK_NAME} {NOTEBOOK_VERSION}) -> chay nen.')
    threading.Thread(target=_run_async, args=(jid, tag, work), daemon=True).start()
    return {'job_id': jid, 'status': 'processing', 'backend': BACKEND,
            'test_case': _slug(test_case or TEST_CASE),
            'result_url': f'/jobs/{jid}/result',
            'notebook': NOTEBOOK_NAME, 'notebook_version': NOTEBOOK_VERSION}


def _active_jobs():
    with JOBS_LOCK:
        return sum(1 for v in JOBS.values() if v.get('status') == 'processing')


def _out_paths(uid, test_case, ext='mp4'):
    fn = f'{uid}_{BACKEND}_{_slug(test_case)}_{NOTEBOOK_VERSION}.{ext}'
    return os.path.join(RES_DIR, fn), fn


STARTED_AT = time.time()


def _json_an_toan(o):
    """Starlette dump JSON voi allow_nan=False -- mot NaN lot vao la endpoint tra HTTP 500."""
    import math
    if isinstance(o, float):
        return o if math.isfinite(o) else str(o)
    if isinstance(o, dict):
        return {k: _json_an_toan(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_json_an_toan(v) for v in o]
    return o


# ============================================================ endpoints
@app.get('/version')
def version():
    try:
        import torch as _t
        gpu = _t.cuda.get_device_name(0) if _t.cuda.is_available() else None
    except Exception:
        gpu = None
    return {
        'notebook': NOTEBOOK_NAME, 'notebook_version': NOTEBOOK_VERSION, 'backend': BACKEND,
        'ai_type': AI_TYPE, 'generate_paths': DUONG_GENERATE, 'test_case': TEST_CASE,
        'models': MODELS, 'changelog': CHANGELOG, 'params': list(BACKEND_PARAMS),
        'mac_dinh': MAC_DINH, 'gpu': gpu, 'models_ready': STATE['models_ready'],
        'uptime_s': round(time.time() - STARTED_AT, 1),
        'jobs': {'total': len(JOBS), 'active': _active_jobs(),
                 'done': sum(1 for v in JOBS.values() if v.get('status') == 'done'),
                 'error': sum(1 for v in JOBS.values() if v.get('status') == 'error')},
    }


@app.get('/health')
def health():
    return {'ok': True, 'notebook': NOTEBOOK_NAME, 'notebook_version': NOTEBOOK_VERSION,
            'backend': BACKEND, 'ai_type': AI_TYPE, 'test_case': TEST_CASE,
            'generate_paths': DUONG_GENERATE,
            'models_ready': STATE['models_ready'], 'active_jobs': _active_jobs(),
            'params': list(BACKEND_PARAMS)}


@app.get('/jobs/{job_id}')
def job_status(job_id: str):
    with JOBS_LOCK:
        j = JOBS.get(job_id)
        if not j:
            raise HTTPException(status_code=404, detail='job not found')
        return _json_an_toan({
            'job_id': job_id, 'status': j['status'], 'stage': j['stage'],
            'progress': j['progress'], 'error': j['error'],
            'elapsed': round(time.time() - j['created'], 1),
            'backend': BACKEND, 'notebook_version': NOTEBOOK_VERSION,
            'test_case': j.get('test_case'), 'filename': j.get('filename'),
            'so_lieu': j.get('so_lieu'),
            'error_type': j.get('error_type'), 'where': j.get('where'),
            'traceback': j.get('traceback')})


@app.get('/jobs/{job_id}/result')
def job_result(job_id: str):
    with JOBS_LOCK:
        j = JOBS.get(job_id)
        if not j:
            raise HTTPException(status_code=404, detail='job not found')
        status, path, fn, err = j['status'], j['result'], j['filename'], j['error']
    if status == 'processing':
        raise HTTPException(status_code=409, detail='job chua xong')
    if status == 'error':
        raise HTTPException(status_code=500, detail=err or 'job error')
    if not path or not os.path.exists(path):
        raise HTTPException(status_code=410, detail='ket qua khong con (da bi don)')
    return FileResponse(path, media_type='video/mp4' if fn.endswith('.mp4') else 'image/png',
                        filename=fn)


@app.get('/laststats')
def laststats():
    return _json_an_toan(LAST_STATS)


@app.get('/', response_class=HTMLResponse)
def trang_chu():
    return (f'<!doctype html><meta charset=utf-8><title>{NOTEBOOK_NAME}</title>'
            f'<body style="font:14px system-ui;max-width:640px;margin:2rem auto">'
            f'<h3>{NOTEBOOK_NAME} {NOTEBOOK_VERSION} &mdash; backend <code>{BACKEND}</code></h3>'
            f'<p>type <code>{AI_TYPE}</code> &middot; <a href=/version>/version</a> &middot; '
            f'<a href=/health>/health</a> &middot; <a href=/laststats>/laststats</a></p>'
            f'<p>POST JSON toi <code>{DUONG_GENERATE[0]}</code>, roi GET /jobs/{{id}} va '
            f'/jobs/{{id}}/result.</p>')


def nhan_job(job: Job):
    uid = uuid.uuid4().hex[:8]
    case = job.test_case or TEST_CASE
    out_path, fn = _out_paths(uid, case, ext='mp4')
    jid = _new_job(job.job_id, job.callback_url, job.callback_token)
    prog = _progress_cb(jid)

    def work():
        return chay_job(job, uid, out_path, progress=prog), fn

    return _spawn(jid, BACKEND, work, case)


# ── Đường mà api.downloadvideo.vn gọi ───────────────────────────────────────
# api dựng đường dẫn THẲNG từ tên type: POST /generate/<type>. Đăng ký CÙNG MỘT hàm
# dưới mọi tên; dict.fromkeys để giữ thứ tự và bỏ trùng.
DUONG_GENERATE = ['/generate/' + t for t in dict.fromkeys([AI_TYPE] + AI_TYPE_ALIAS)]
for _d in DUONG_GENERATE:
    app.post(_d)(nhan_job)


# ==== KHOI DONG ====  (test dung o day: phia duoi can GPU, mang, Colab)
print(f'>>> [{NOTEBOOK_NAME} {NOTEBOOK_VERSION}] backend={BACKEND} case={TEST_CASE}')
_ok = True
for _name, _fn in PRELOAD:
    _t0 = time.time()
    try:
        _fn()
        print(f'   [preload] {_name} OK ({time.time() - _t0:.0f}s)')
    except Exception as _e:
        _ok = False
        traceback.print_exc()
        print(f'   [preload] {_name} LOI: {_e}')
STATE['models_ready'] = _ok
print('>>> MODEL SAN SANG.' if _ok else '>>> MODEL CHUA SAN SANG (xem log tren).')


def _port_in_use(p):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('127.0.0.1', p)) == 0


if _port_in_use(PORT):
    print(f'uvicorn da chay san o port {PORT} (chay lai cell) -> KHONG khoi dong lai.')
else:
    threading.Thread(target=lambda: uvicorn.run(app, host='0.0.0.0', port=PORT,
                                                log_level='warning'), daemon=True).start()
    time.sleep(3)

try:
    subprocess.run(['pkill', '-f', 'cloudflared tunnel'],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(1)
except Exception:
    pass

# Bao PUBLIC_URL len api: bot bom window.__ai_boot truoc khi mo notebook.
AI_API_BASE = 'https://api.downloadvideo.vn'


def _read_ai_boot(tries=6, wait=5):
    import json as _json
    try:
        from google.colab import output as _out
    except Exception as e:
        print('   [report] khong co google.colab.output:', e)
        return None
    for i in range(1, tries + 1):
        try:
            raw = _out.eval_js('JSON.stringify(window.__ai_boot || null)', timeout_sec=20)
            boot = _json.loads(raw) if raw and raw != 'null' else None
            if boot and boot.get('account') and boot.get('token'):
                return boot
            print(f'   [report] lan {i}/{tries}: chua thay window.__ai_boot')
        except Exception as e:
            print(f'   [report] lan {i}/{tries}: eval_js loi: {e}')
        time.sleep(wait)
    return None


def _report_public_url(url):
    boot = _read_ai_boot()
    if not boot:
        print('   [report] KHONG co __ai_boot (chay tay?) -> bot se quet dong PUBLIC_URL=.')
        return False
    api = (boot.get('api') or AI_API_BASE).rstrip('/')
    body = {'account': boot['account'], 'token': boot['token'], 'url': url}
    hdr = {'ngrok-skip-browser-warning': '1', 'User-Agent': 'colab-tunnel-report/1.0'}
    for attempt in range(1, 25):
        try:
            r = requests.post(f'{api}/api/c/ai/profiles/tunnel', json=body, headers=hdr, timeout=20)
            print(f'   [report] {attempt}/24 -> {r.status_code} {r.text[:200]}')
            if r.status_code == 200 and r.json().get('success'):
                return True
            if r.status_code in (400, 401):
                return False
        except Exception as e:
            print(f'   [report] {attempt}/24 loi mang: {e}')
        time.sleep(15)
    return False


proc = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

public_url = None
for line in proc.stdout:
    print(line, end='')
    m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
    if m and not public_url:
        public_url = m.group(0)
        print('\n\nPUBLIC_URL=' + public_url + '\n', flush=True)
        break

if public_url:
    threading.Thread(target=_report_public_url, args=(public_url,), daemon=True).start()

print(f'=== {NOTEBOOK_NAME} {NOTEBOOK_VERSION} | backend={BACKEND} | case={TEST_CASE} ===')
print('URL:', public_url)
print('Endpoints: GET /version | GET /health | POST ' + DUONG_GENERATE[0] + ' (JSON)')
print('           GET /jobs/{id} | GET /jobs/{id}/result | GET /laststats')
print('>>> SAN SANG. Giu cell nay chay.')
while True:
    line = proc.stdout.readline()
    if not line:
        break
    if 'ERR' in line or 'error' in line.lower():
        print(line, end='')


>>> [ImageVideoToVideo_WanVACE_Colab.ipynb v5] backend=wan_vace_dk case=V5-41f-8fps-512x896-Q5KM-lightx2v-densepose
   [wan_vace_dk] du 6 file model, GPU: Tesla T4
   [preload] kiem_san_sang OK (0s)
>>> MODEL SAN SANG.
2026-09-17T01:34:16Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-17T01:34:16Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-17T01:34:19Z INF +----------------------------------------------------------------------------

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `hf_hub_download`. Downloads always resume whenever possible.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


densepose_r50_fpn_dl.torchscript: reconstructing file:   0%|          |  0.00B /  265MB            

densepose_r50_fpn_dl.torchscript: downloading bytes:           |  0.00B            

[Errno 2] No such file or directory: '/tmp/ckpts'
model_path is /content/ComfyUI/custom_nodes/comfyui_controlnet_aux/ckpts/LayerNorm/DensePose-TorchScript-with-hint-image/densepose_r50_fpn_dl.torchscript


/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py:1750: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:3637.)
  return forward_call(*args, **kwargs)


   [wan_vace_dk] ma_hoa_vace ...
   [wan_vace_dk] nap_model ...
gguf qtypes: F32 (836), Q5_K (345), Q6_K (144), F16 (6)


/content/ComfyUI/custom_nodes/ComfyUI_GGUF/loader.py:91: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:203.)
  torch_tensor = torch.from_numpy(tensor.data) # mmap


   [wan_vace_dk] LoRA lightx2v cuong do 1.0
   [wan_vace_dk] denoise ...
Attempting to release mmap (151)


  0%|          | 0/4 [00:00<?, ?it/s]